# 08. Small LLM Student Distillation Experiment

Отдельный эксперимент по сравнению двух student-подходов для teacher-student distillation в задаче бинарной классификации риск-сигналов в банковских новостях:

1. `cointegrated/rubert-tiny2` как compact encoder-transformer student;
2. `Qwen/Qwen2.5-0.5B` как small decoder-only LLM student / sequence classification student.

Основной notebook `07_postprocessing_and_error_analysis.ipynb` не изменяется. Цель эксперимента - методически проверить более выразительные student-классы по сравнению с TF-IDF student, а не обязательно улучшить финальные метрики проекта.


## 1. Imports and setup


In [5]:
!pip uninstall -y torchao torchaudio torchtext torchvision
!pip install -q "pandas==2.2.2" "numpy==2.0.2"
!pip install -q -U transformers accelerate peft bitsandbytes scikit-learn

In [ ]:
import os
os.kill(os.getpid(), 9)

In [3]:
from pathlib import Path
import inspect
import random
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

warnings.filterwarnings("ignore")

SEED = 42

RUN_RUBERT_TINY2 = True
RUN_QWEN_STUDENT = True
USE_LORA_FOR_QWEN = True

RUBERT_MODEL_CANDIDATES = [
    "cointegrated/rubert-tiny2",
    "cointegrated/rubert-tiny",
]

QWEN_MODEL_CANDIDATES = [
    "Qwen/Qwen2.5-0.5B",
    "Qwen/Qwen2.5-0.5B-Instruct",
]

TARGET_COLUMN = "alert_flag"
TEXT_COLUMNS = ["title", "entity_norm", "text_fragment"]


# =========================
# Project / Colab paths
# =========================

PROJECT_ROOT = Path.cwd()

# Если ноутбук открыт в Colab, рабочая директория обычно /content
if Path("/content").exists():
    PROJECT_ROOT = Path("/content")

# Если ноутбук лежит в папке notebooks внутри проекта
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]


def first_existing_path(candidates, name, required=True):
    """
    Ищет первый существующий путь из списка.
    Поддерживает и структуру проекта, и flat-загрузку файлов в Colab.
    """
    for path in candidates:
        path = Path(path)
        if path.exists():
            print(f"{name}: {path}")
            return path

    message = f"{name} not found. Checked:\n" + "\n".join(str(Path(p)) for p in candidates)

    if required:
        raise FileNotFoundError(message)

    print("WARNING:", message)
    return None


DATA_PATH = first_existing_path(
    [
        PROJECT_ROOT / "homework_04_dataset" / "data" / "dataset_for_training.csv",
        PROJECT_ROOT / "dataset_for_training.csv",
    ],
    "DATA_PATH",
)

TRAIN_TEACHER_V2B_PATH = first_existing_path(
    [
        PROJECT_ROOT / "homework_07_postprocessing" / "reports" / "teacher_labels" / "train_teacher_prompt_v2b.csv",
        PROJECT_ROOT / "train_teacher_prompt_v2b.csv",
    ],
    "TRAIN_TEACHER_V2B_PATH",
    required=False,
)

TRAIN_TEACHER_V1_PATH = first_existing_path(
    [
        PROJECT_ROOT / "homework_07_postprocessing" / "reports" / "teacher_labels" / "train_teacher_prompt_v1.csv",
        PROJECT_ROOT / "train_teacher_prompt_v1.csv",
    ],
    "TRAIN_TEACHER_V1_PATH",
    required=False,
)

if TRAIN_TEACHER_V2B_PATH is None and TRAIN_TEACHER_V1_PATH is None:
    raise FileNotFoundError(
        "No teacher label file found. Upload either "
        "`train_teacher_prompt_v2b.csv` or `train_teacher_prompt_v1.csv`."
    )

HW7_DIR = PROJECT_ROOT / "homework_07_postprocessing"
TEACHER_DIR = HW7_DIR / "reports" / "teacher_labels"
STUDENT_PRED_DIR = HW7_DIR / "reports" / "student_predictions"
ERROR_DIR = HW7_DIR / "reports" / "error_analysis"
MODEL_DIR = HW7_DIR / "models"

for path in [TEACHER_DIR, STUDENT_PRED_DIR, ERROR_DIR, MODEL_DIR]:
    path.mkdir(parents=True, exist_ok=True)


# =========================
# Reproducibility
# =========================

random.seed(SEED)
np.random.seed(SEED)


# =========================
# Torch / Transformers
# =========================

try:
    import torch
    from torch import nn
    from torch.utils.data import Dataset

    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
        EarlyStoppingCallback,
        Trainer,
        TrainingArguments,
        set_seed,
    )

    TRANSFORMERS_AVAILABLE = True
    set_seed(SEED)

except ImportError as exc:
    TRANSFORMERS_AVAILABLE = False
    torch = None
    print("torch/transformers are not available. Model experiments will be skipped.")
    print(repr(exc))


# =========================
# PEFT / LoRA
# =========================

try:
    from peft import LoraConfig, TaskType, get_peft_model

    PEFT_AVAILABLE = True

except ImportError as exc:
    PEFT_AVAILABLE = False
    print("peft is not available. Qwen LoRA will fall back to classifier-head training if Qwen is run.")
    print(repr(exc))


# =========================
# Device
# =========================

if TRANSFORMERS_AVAILABLE:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", DEVICE)

    if DEVICE != "cuda":
        print(
            "Warning: GPU is not available. "
            "Qwen experiment will be skipped by default because it is likely too slow/heavy on CPU."
        )
else:
    DEVICE = "unavailable"


pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)

DATA_PATH: /content/dataset_for_training.csv
TRAIN_TEACHER_V2B_PATH: /content/train_teacher_prompt_v2b.csv
TRAIN_TEACHER_V1_PATH: /content/train_teacher_prompt_v1.csv
Device: cuda


## 2. Load data


In [6]:
df = pd.read_csv(DATA_PATH)

required_columns = ["sample_id", "title", "entity_norm", "text_fragment", TARGET_COLUMN, "split"]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df["model_text"] = (
    df["title"].fillna("").astype(str) + "\n" +
    df["entity_norm"].fillna("").astype(str) + "\n" +
    df["text_fragment"].fillna("").astype(str)
)

train_df = df[df["split"] == "train"].copy()
valid_df = df[df["split"] == "valid"].copy()
test_df = df[df["split"] == "test"].copy()

y_valid = valid_df[TARGET_COLUMN].astype(int).to_numpy()
y_test = test_df[TARGET_COLUMN].astype(int).to_numpy()

print("Dataset shape:", df.shape)
print("Train/valid/test:", len(train_df), len(valid_df), len(test_df))
display(pd.crosstab(df["split"], df[TARGET_COLUMN], margins=True))
display(df[["sample_id", "split", "title", "entity_norm", "text_fragment", TARGET_COLUMN, "model_text"]].head(3))


Dataset shape: (839, 13)
Train/valid/test: 587 126 126


alert_flag,0,1,All
split,,,
test,88,38,126
train,409,178,587
valid,88,38,126
All,585,254,839


,sample_id,split,title,entity_norm,text_fragment,alert_flag,model_text
0,ml_000001,train,"Газпромбанк и РСХБ самоустранились от проекта ""Едим дома!""",Росбанк,"Газпромбанк и РСХБ самоустранились от проекта ""Едим дома!"" РСХБ и ГПБ заявили, что проект не соответствует их действующим кредитным программам. Росбанк (входит в группу Societe...",0,"Газпромбанк и РСХБ самоустранились от проекта ""Едим дома!""\nРосбанк\nГазпромбанк и РСХБ самоустранились от проекта ""Едим дома!"" РСХБ и ГПБ заявили, что проект не соответствует ..."
1,ml_000002,test,Хакеры четыре года грабили клиентов Сбербанка и ВТБ,ВТБ,"Хакеры четыре года грабили клиентов Сбербанка и ВТБ Во время задержания хакеры попытались смыть в унитаз деньги, флешки и телефоны, а также хотели уничтожить компьютерную техни...",1,"Хакеры четыре года грабили клиентов Сбербанка и ВТБ\nВТБ\nХакеры четыре года грабили клиентов Сбербанка и ВТБ Во время задержания хакеры попытались смыть в унитаз деньги, флешк..."
2,ml_000003,train,"Exxon и ""Роснефть"" прекратили бурение в Арктике из-за санкций",Газпромбанк,"Exxon и ""Роснефть"" прекратили бурение в Арктике из-за санкций Вместе с тем страны ЕС расширили и индивидуальные ограничительные меры, добавив в санкционный список 24 физических...",0,"Exxon и ""Роснефть"" прекратили бурение в Арктике из-за санкций\nГазпромбанк\nExxon и ""Роснефть"" прекратили бурение в Арктике из-за санкций Вместе с тем страны ЕС расширили и инд..."


## 3. Load teacher labels


In [7]:
# =========================
# Load teacher labels
# =========================

def first_existing_path(candidates, name, required=True):
    for path in candidates:
        path = Path(path)
        if path.exists():
            print(f"{name}: {path}")
            return path

    message = f"{name} not found. Checked:\n" + "\n".join(str(Path(p)) for p in candidates)

    if required:
        raise FileNotFoundError(message)

    print("WARNING:", message)
    return None


TEACHER_V2B_PATH = first_existing_path(
    [
        TEACHER_DIR / "train_teacher_prompt_v2b.csv",
        PROJECT_ROOT / "train_teacher_prompt_v2b.csv",
        Path("/content/train_teacher_prompt_v2b.csv"),
    ],
    "TEACHER_V2B_PATH",
    required=False,
)

TEACHER_V1_PATH = first_existing_path(
    [
        TEACHER_DIR / "train_teacher_prompt_v1.csv",
        PROJECT_ROOT / "train_teacher_prompt_v1.csv",
        Path("/content/train_teacher_prompt_v1.csv"),
    ],
    "TEACHER_V1_PATH",
    required=False,
)

if TEACHER_V2B_PATH is not None:
    teacher_path = TEACHER_V2B_PATH
    teacher_source = "teacher_prompt_v2b"
elif TEACHER_V1_PATH is not None:
    teacher_path = TEACHER_V1_PATH
    teacher_source = "teacher_prompt_v1"
else:
    raise FileNotFoundError(
        "Teacher labels not found. Upload one of these files:\n"
        "- train_teacher_prompt_v2b.csv\n"
        "- train_teacher_prompt_v1.csv"
    )

teacher_df = pd.read_csv(teacher_path)

required_teacher_columns = ["sample_id", "teacher_label", "teacher_confidence"]
missing_teacher_columns = [
    col for col in required_teacher_columns
    if col not in teacher_df.columns
]

if missing_teacher_columns:
    raise ValueError(
        f"Missing teacher columns in {teacher_path}: {missing_teacher_columns}\n"
        f"Available columns: {teacher_df.columns.tolist()}"
    )

teacher_train_df = train_df[["sample_id", "model_text", TARGET_COLUMN]].merge(
    teacher_df[required_teacher_columns].copy(),
    on="sample_id",
    how="inner",
    validate="one_to_one",
)

teacher_train_df = teacher_train_df[
    teacher_train_df["teacher_label"].isin([0, 1])
    & teacher_train_df["teacher_confidence"].notna()
].copy()

teacher_train_df["teacher_label"] = teacher_train_df["teacher_label"].astype(int)
teacher_train_df["teacher_confidence"] = (
    teacher_train_df["teacher_confidence"]
    .astype(float)
    .clip(0.0, 1.0)
)

teacher_train_df["soft_y"] = np.where(
    teacher_train_df["teacher_label"] == 1,
    teacher_train_df["teacher_confidence"],
    1.0 - teacher_train_df["teacher_confidence"],
).clip(0.0, 1.0)

teacher_train_df["teacher_hard_y"] = teacher_train_df["teacher_label"].astype(int)

print("Teacher source:", teacher_source)
print("Teacher path:", teacher_path)
print("Teacher shape:", teacher_df.shape)
print("Train rows with valid teacher labels:", len(teacher_train_df), "/", len(train_df))

display(
    teacher_train_df[
        ["sample_id", TARGET_COLUMN, "teacher_label", "teacher_confidence", "soft_y"]
    ].head()
)

display(
    teacher_train_df["teacher_label"]
    .value_counts()
    .rename("teacher_label_count")
)

TEACHER_V2B_PATH: /content/train_teacher_prompt_v2b.csv
TEACHER_V1_PATH: /content/train_teacher_prompt_v1.csv
Teacher source: teacher_prompt_v2b
Teacher path: /content/train_teacher_prompt_v2b.csv
Teacher shape: (587, 5)
Train rows with valid teacher labels: 587 / 587


,sample_id,alert_flag,teacher_label,teacher_confidence,soft_y
0,ml_000001,0,0,0.8,0.2
1,ml_000003,0,1,0.9,0.9
2,ml_000005,0,0,0.9,0.1
3,ml_000007,0,0,0.6,0.4
4,ml_000008,0,0,0.3,0.7


,teacher_label_count
teacher_label,
0,348
1,239


## 4. Metrics helpers


In [8]:
def compute_binary_metrics(y_true, y_pred, y_score=None):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
    }
    if y_score is not None and len(np.unique(y_true)) == 2:
        metrics["roc_auc"] = roc_auc_score(y_true, y_score)
        metrics["pr_auc"] = average_precision_score(y_true, y_score)
    else:
        metrics["roc_auc"] = np.nan
        metrics["pr_auc"] = np.nan
    return metrics


def select_best_threshold_by_f1(y_true, y_score, thresholds=np.arange(0.1, 0.91, 0.05)):
    rows = []
    for threshold in thresholds:
        y_pred = (np.asarray(y_score) >= threshold).astype(int)
        rows.append({"threshold": round(float(threshold), 2), **compute_binary_metrics(y_true, y_pred, y_score)})
    threshold_table = pd.DataFrame(rows)
    best_row = threshold_table.sort_values(["f1", "recall", "precision"], ascending=False).iloc[0]
    return float(best_row["threshold"]), threshold_table


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def positive_scores_from_logits(logits):
    logits = np.asarray(logits)
    if logits.ndim == 2 and logits.shape[1] == 2:
        return sigmoid(logits[:, 1] - logits[:, 0])
    if logits.ndim == 2 and logits.shape[1] == 1:
        return sigmoid(logits[:, 0])
    return sigmoid(logits.reshape(-1))


def append_result_row(rows, model, student_type, distillation_mode, dataset, threshold, metrics, notes=""):
    rows.append({
        "model": model,
        "student_type": student_type,
        "teacher_source": teacher_source,
        "distillation_mode": distillation_mode,
        "dataset": dataset,
        "threshold": threshold,
        "precision": metrics.get("precision", np.nan),
        "recall": metrics.get("recall", np.nan),
        "f1": metrics.get("f1", np.nan),
        "roc_auc": metrics.get("roc_auc", np.nan),
        "pr_auc": metrics.get("pr_auc", np.nan),
        "tp": metrics.get("tp", np.nan),
        "fp": metrics.get("fp", np.nan),
        "fn": metrics.get("fn", np.nan),
        "tn": metrics.get("tn", np.nan),
        "notes": notes,
    })


## 5. Baseline reference


In [9]:
try:
    pd
except NameError:
    import pandas as pd

try:
    np
except NameError:
    import numpy as np

reference_metrics = pd.DataFrame([
    {
        "model": "TF-IDF student v2b final",
        "student_type": "TF-IDF + LogisticRegression",
        "teacher_source": "teacher_prompt_v2b",
        "distillation_mode": "soft labels + TF-IDF student",
        "dataset": "valid",
        "threshold": 0.50,
        "precision": 0.738,
        "recall": 0.816,
        "f1": 0.775,
        "roc_auc": 0.910,
        "pr_auc": 0.824,
        "tp": np.nan,
        "fp": np.nan,
        "fn": np.nan,
        "tn": np.nan,
        "notes": "Reference from current project notebook",
    },
    {
        "model": "TF-IDF student v2b final",
        "student_type": "TF-IDF + LogisticRegression",
        "teacher_source": "teacher_prompt_v2b",
        "distillation_mode": "soft labels + TF-IDF student",
        "dataset": "test",
        "threshold": 0.50,
        "precision": 0.773,
        "recall": 0.895,
        "f1": 0.829,
        "roc_auc": 0.969,
        "pr_auc": 0.936,
        "tp": np.nan,
        "fp": np.nan,
        "fn": np.nan,
        "tn": np.nan,
        "notes": "Reference from current project notebook",
    },
    {
        "model": "rubert-tiny2 previous run",
        "student_type": "compact encoder-transformer",
        "teacher_source": "teacher_prompt_v2b",
        "distillation_mode": "soft labels",
        "dataset": "valid",
        "threshold": np.nan,
        "precision": 0.400,
        "recall": 0.895,
        "f1": 0.553,
        "roc_auc": 0.772,
        "pr_auc": 0.607,
        "tp": np.nan,
        "fp": np.nan,
        "fn": np.nan,
        "tn": np.nan,
        "notes": "Previous rubert-tiny2 run from current project context",
    },
    {
        "model": "rubert-tiny2 previous run",
        "student_type": "compact encoder-transformer",
        "teacher_source": "teacher_prompt_v2b",
        "distillation_mode": "soft labels",
        "dataset": "test",
        "threshold": np.nan,
        "precision": 0.440,
        "recall": 0.868,
        "f1": 0.584,
        "roc_auc": 0.754,
        "pr_auc": 0.585,
        "tp": np.nan,
        "fp": np.nan,
        "fn": np.nan,
        "tn": np.nan,
        "notes": "Previous rubert-tiny2 run from current project context",
    },
])

display(reference_metrics)


,model,student_type,teacher_source,distillation_mode,dataset,threshold,precision,recall,f1,roc_auc,pr_auc,tp,fp,fn,tn,notes
0,TF-IDF student v2b final,TF-IDF + LogisticRegression,teacher_prompt_v2b,soft labels + TF-IDF student,valid,0.5,0.738,0.816,0.775,0.910,0.824,NaN,NaN,NaN,NaN,Reference from current project notebook
1,TF-IDF student v2b final,TF-IDF + LogisticRegression,teacher_prompt_v2b,soft labels + TF-IDF student,test,0.5,0.773,0.895,0.829,0.969,0.936,NaN,NaN,NaN,NaN,Reference from current project notebook
2,rubert-tiny2 previous run,compact encoder-transformer,teacher_prompt_v2b,soft labels,valid,NaN,0.400,0.895,0.553,0.772,0.607,NaN,NaN,NaN,NaN,Previous rubert-tiny2 run from current project context
3,rubert-tiny2 previous run,compact encoder-transformer,teacher_prompt_v2b,soft labels,test,NaN,0.440,0.868,0.584,0.754,0.585,NaN,NaN,NaN,NaN,Previous rubert-tiny2 run from current project context


## Shared Trainer utilities


In [10]:
if "compute_binary_metrics" not in globals():
    def compute_binary_metrics(y_true, y_pred, y_score=None):
        y_true = np.asarray(y_true).astype(int)
        y_pred = np.asarray(y_pred).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        metrics = {
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "tp": int(tp),
            "fp": int(fp),
            "fn": int(fn),
            "tn": int(tn),
        }
        if y_score is not None and len(np.unique(y_true)) == 2:
            metrics["roc_auc"] = roc_auc_score(y_true, y_score)
            metrics["pr_auc"] = average_precision_score(y_true, y_score)
        else:
            metrics["roc_auc"] = np.nan
            metrics["pr_auc"] = np.nan
        return metrics


if "select_best_threshold_by_f1" not in globals():
    def select_best_threshold_by_f1(y_true, y_score, thresholds=np.arange(0.1, 0.91, 0.05)):
        rows = []
        for threshold in thresholds:
            y_pred = (np.asarray(y_score) >= threshold).astype(int)
            rows.append({"threshold": round(float(threshold), 2), **compute_binary_metrics(y_true, y_pred, y_score)})
        threshold_table = pd.DataFrame(rows)
        best_row = threshold_table.sort_values(["f1", "recall", "precision"], ascending=False).iloc[0]
        return float(best_row["threshold"]), threshold_table


if "sigmoid" not in globals():
    def sigmoid(x):
        return 1.0 / (1.0 + np.exp(-x))


if "positive_scores_from_logits" not in globals():
    def positive_scores_from_logits(logits):
        logits = np.asarray(logits)
        if logits.ndim == 2 and logits.shape[1] == 2:
            return sigmoid(logits[:, 1] - logits[:, 0])
        if logits.ndim == 2 and logits.shape[1] == 1:
            return sigmoid(logits[:, 0])
        return sigmoid(logits.reshape(-1))


if "append_result_row" not in globals():
    def append_result_row(rows, model, student_type, distillation_mode, dataset, threshold, metrics, notes=""):
        rows.append({
            "model": model,
            "student_type": student_type,
            "teacher_source": teacher_source,
            "distillation_mode": distillation_mode,
            "dataset": dataset,
            "threshold": threshold,
            "precision": metrics.get("precision", np.nan),
            "recall": metrics.get("recall", np.nan),
            "f1": metrics.get("f1", np.nan),
            "roc_auc": metrics.get("roc_auc", np.nan),
            "pr_auc": metrics.get("pr_auc", np.nan),
            "tp": metrics.get("tp", np.nan),
            "fp": metrics.get("fp", np.nan),
            "fn": metrics.get("fn", np.nan),
            "tn": metrics.get("tn", np.nan),
            "notes": notes,
        })


if TRANSFORMERS_AVAILABLE:
    class NewsRiskDataset(Dataset):
        def __init__(self, tokenizer, texts, labels=None, max_length=256):
            self.encodings = tokenizer(
                list(texts),
                truncation=True,
                padding=True,
                max_length=max_length,
            )
            self.labels = None if labels is None else np.asarray(labels, dtype=np.float32)

        def __len__(self):
            return len(self.encodings["input_ids"])

        def __getitem__(self, idx):
            item = {key: torch.tensor(value[idx]) for key, value in self.encodings.items()}
            if self.labels is not None:
                item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)
            return item


    class SoftLabelTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            logits = outputs.logits
            if logits.ndim == 2 and logits.shape[1] == 2:
                positive_logit = logits[:, 1] - logits[:, 0]
            else:
                positive_logit = logits.reshape(-1)
            loss = nn.BCEWithLogitsLoss()(positive_logit, labels.float())
            return (loss, outputs) if return_outputs else loss


    def compute_transformer_metrics(eval_pred):
        logits, labels = eval_pred
        y_true = np.asarray(labels).astype(int)
        y_score = positive_scores_from_logits(logits)
        y_pred = (y_score >= 0.5).astype(int)
        metrics = compute_binary_metrics(y_true, y_pred, y_score)
        return {
            "accuracy": metrics["accuracy"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1": metrics["f1"],
            "roc_auc": metrics["roc_auc"],
            "pr_auc": metrics["pr_auc"],
        }


    def make_training_args(
        output_dir,
        learning_rate,
        train_batch_size,
        eval_batch_size,
        num_train_epochs,
        weight_decay=0.01,
        gradient_accumulation_steps=1,
        fp16=False,
        bf16=False,
    ):
        signature = inspect.signature(TrainingArguments.__init__)
        strategy_arg = "eval_strategy" if "eval_strategy" in signature.parameters else "evaluation_strategy"
        kwargs = {
            "output_dir": str(output_dir),
            strategy_arg: "epoch",
            "save_strategy": "epoch",
            "learning_rate": learning_rate,
            "per_device_train_batch_size": train_batch_size,
            "per_device_eval_batch_size": eval_batch_size,
            "gradient_accumulation_steps": gradient_accumulation_steps,
            "num_train_epochs": num_train_epochs,
            "weight_decay": weight_decay,
            "load_best_model_at_end": True,
            "metric_for_best_model": "f1",
            "greater_is_better": True,
            "seed": SEED,
            "logging_steps": 20,
            "report_to": "none",
            "fp16": bool(fp16),
            "bf16": bool(bf16),
        }
        if "save_total_limit" in signature.parameters:
            kwargs["save_total_limit"] = 1
        return TrainingArguments(**kwargs)


    def load_first_available_sequence_classifier(model_candidates, num_labels=2):
        last_error = None
        for model_name in model_candidates:
            try:
                tokenizer = AutoTokenizer.from_pretrained(model_name)
                model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
                return model_name, tokenizer, model
            except Exception as exc:
                last_error = exc
                print(f"Could not load {model_name}: {exc}")
        raise RuntimeError(f"No model could be loaded from candidates {model_candidates}. Last error: {last_error}")


def ensure_distillation_data_loaded():
    global df, train_df, valid_df, test_df, y_valid, y_test
    global teacher_df, teacher_train_df, teacher_path, teacher_source

    dataset_missing = any(name not in globals() for name in ["df", "train_df", "valid_df", "test_df", "y_valid", "y_test"])
    if not dataset_missing and "model_text" not in df.columns:
        dataset_missing = True

    if dataset_missing:
        df = pd.read_csv(DATA_PATH)
        required_columns = ["sample_id", "title", "entity_norm", "text_fragment", TARGET_COLUMN, "split"]
        missing_columns = [col for col in required_columns if col not in df.columns]
        if missing_columns:
            raise ValueError(f"Missing required columns: {missing_columns}")

        df["model_text"] = (
            df["title"].fillna("").astype(str) + "\n" +
            df["entity_norm"].fillna("").astype(str) + "\n" +
            df["text_fragment"].fillna("").astype(str)
        )
        train_df = df[df["split"] == "train"].copy()
        valid_df = df[df["split"] == "valid"].copy()
        test_df = df[df["split"] == "test"].copy()
        y_valid = valid_df[TARGET_COLUMN].astype(int).to_numpy()
        y_test = test_df[TARGET_COLUMN].astype(int).to_numpy()
        print("Loaded dataset for experiment cells:", df.shape)

    teacher_missing = "teacher_train_df" not in globals() or teacher_train_df.empty
    if teacher_missing:
        teacher_v2b_path = TEACHER_DIR / "train_teacher_prompt_v2b.csv"
        teacher_v1_path = TEACHER_DIR / "train_teacher_prompt_v1.csv"
        if teacher_v2b_path.exists():
            teacher_path = teacher_v2b_path
            teacher_source = "teacher_prompt_v2b"
        elif teacher_v1_path.exists():
            teacher_path = teacher_v1_path
            teacher_source = "teacher_prompt_v1"
        else:
            raise FileNotFoundError(f"Teacher labels not found: {teacher_v2b_path} or {teacher_v1_path}")

        teacher_df = pd.read_csv(teacher_path)
        required_teacher_columns = ["sample_id", "teacher_label", "teacher_confidence"]
        missing_teacher_columns = [col for col in required_teacher_columns if col not in teacher_df.columns]
        if missing_teacher_columns:
            raise ValueError(f"Missing teacher columns in {teacher_path}: {missing_teacher_columns}")

        teacher_train_df = train_df[["sample_id", "model_text", TARGET_COLUMN]].merge(
            teacher_df[required_teacher_columns].copy(),
            on="sample_id",
            how="inner",
            validate="one_to_one",
        )
        teacher_train_df = teacher_train_df[
            teacher_train_df["teacher_label"].isin([0, 1]) & teacher_train_df["teacher_confidence"].notna()
        ].copy()
        teacher_train_df["teacher_label"] = teacher_train_df["teacher_label"].astype(int)
        teacher_train_df["teacher_confidence"] = teacher_train_df["teacher_confidence"].astype(float).clip(0.0, 1.0)
        teacher_train_df["soft_y"] = np.where(
            teacher_train_df["teacher_label"] == 1,
            teacher_train_df["teacher_confidence"],
            1.0 - teacher_train_df["teacher_confidence"],
        ).clip(0.0, 1.0)
        teacher_train_df["teacher_hard_y"] = teacher_train_df["teacher_label"].astype(int)
        print("Loaded teacher labels for experiment cells:", len(teacher_train_df), "/", len(train_df))

    return teacher_train_df


experiment_rows = []
skipped_rows = []


## 6. Experiment A: rubert-tiny2 student


In [11]:
rubert_valid_metrics = pd.DataFrame()
rubert_test_metrics = pd.DataFrame()
rubert_threshold_table = pd.DataFrame()

if not RUN_RUBERT_TINY2:
    skipped_rows.append({"experiment": "rubert-tiny2 student", "status": "skipped", "reason": "RUN_RUBERT_TINY2=False"})
elif not TRANSFORMERS_AVAILABLE:
    skipped_rows.append({"experiment": "rubert-tiny2 student", "status": "skipped", "reason": "torch/transformers are not installed"})
else:
    ensure_distillation_data_loaded()

    rubert_model_name, rubert_tokenizer, rubert_model = load_first_available_sequence_classifier(RUBERT_MODEL_CANDIDATES, num_labels=2)
    print("Rubert student model:", rubert_model_name)

    rubert_train_dataset = NewsRiskDataset(
        rubert_tokenizer,
        teacher_train_df["model_text"],
        labels=teacher_train_df["soft_y"],
        max_length=256,
    )
    rubert_valid_dataset = NewsRiskDataset(
        rubert_tokenizer,
        valid_df["model_text"],
        labels=y_valid,
        max_length=256,
    )
    rubert_test_dataset = NewsRiskDataset(
        rubert_tokenizer,
        test_df["model_text"],
        labels=y_test,
        max_length=256,
    )

    rubert_args = make_training_args(
        output_dir=MODEL_DIR / "rubert_tiny2_student",
        learning_rate=2e-5,
        train_batch_size=8,
        eval_batch_size=16,
        num_train_epochs=2,
        weight_decay=0.01,
    )

    rubert_trainer = SoftLabelTrainer(
        model=rubert_model,
        args=rubert_args,
        train_dataset=rubert_train_dataset,
        eval_dataset=rubert_valid_dataset,
        compute_metrics=compute_transformer_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
    )

    rubert_trainer.train()

    rubert_valid_logits = rubert_trainer.predict(rubert_valid_dataset).predictions
    rubert_test_logits = rubert_trainer.predict(rubert_test_dataset).predictions
    rubert_valid_score = positive_scores_from_logits(rubert_valid_logits)
    rubert_test_score = positive_scores_from_logits(rubert_test_logits)

    rubert_threshold, rubert_threshold_table = select_best_threshold_by_f1(y_valid, rubert_valid_score)
    rubert_valid_pred = (rubert_valid_score >= rubert_threshold).astype(int)
    rubert_test_pred = (rubert_test_score >= rubert_threshold).astype(int)

    rubert_valid_metric_dict = compute_binary_metrics(y_valid, rubert_valid_pred, rubert_valid_score)
    rubert_test_metric_dict = compute_binary_metrics(y_test, rubert_test_pred, rubert_test_score)

    append_result_row(
        experiment_rows,
        model="rubert-tiny2 student",
        student_type="compact encoder-transformer",
        distillation_mode="soft labels via BCEWithLogitsLoss",
        dataset="valid",
        threshold=rubert_threshold,
        metrics=rubert_valid_metric_dict,
        notes=f"model={rubert_model_name}; threshold selected on valid",
    )
    append_result_row(
        experiment_rows,
        model="rubert-tiny2 student",
        student_type="compact encoder-transformer",
        distillation_mode="soft labels via BCEWithLogitsLoss",
        dataset="test",
        threshold=rubert_threshold,
        metrics=rubert_test_metric_dict,
        notes=f"model={rubert_model_name}; valid-selected threshold applied once to test",
    )

    rubert_valid_predictions = valid_df[["sample_id", TARGET_COLUMN]].copy()
    rubert_valid_predictions["student_score"] = rubert_valid_score
    rubert_valid_predictions["student_pred"] = rubert_valid_pred
    rubert_valid_predictions.to_csv(STUDENT_PRED_DIR / "valid_rubert_tiny2_student.csv", index=False)

    rubert_test_predictions = test_df[["sample_id", TARGET_COLUMN]].copy()
    rubert_test_predictions["student_score"] = rubert_test_score
    rubert_test_predictions["student_pred"] = rubert_test_pred
    rubert_test_predictions.to_csv(STUDENT_PRED_DIR / "test_rubert_tiny2_student.csv", index=False)

    rubert_threshold_table.to_csv(ERROR_DIR / "valid_rubert_tiny2_threshold_tuning.csv", index=False)

    rubert_valid_metrics = pd.DataFrame([experiment_rows[-2]])
    rubert_test_metrics = pd.DataFrame([experiment_rows[-1]])
    display(rubert_threshold_table)
    display(pd.concat([rubert_valid_metrics, rubert_test_metrics], ignore_index=True))


config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.74M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc,Pr Auc
1,0.625208,0.857225,0.301587,0.301587,1.000000,0.463415,0.773325,0.591541
2,0.632205,0.851438,0.301587,0.301587,1.000000,0.463415,0.766447,0.593390


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

,threshold,accuracy,precision,recall,f1,tp,fp,fn,tn,roc_auc,pr_auc
0,0.10,0.301587,0.301587,1.000000,0.463415,38,88,0,0,0.773325,0.591541
1,0.15,0.301587,0.301587,1.000000,0.463415,38,88,0,0,0.773325,0.591541
2,0.20,0.301587,0.301587,1.000000,0.463415,38,88,0,0,0.773325,0.591541
3,0.25,0.301587,0.301587,1.000000,0.463415,38,88,0,0,0.773325,0.591541
4,0.30,0.301587,0.301587,1.000000,0.463415,38,88,0,0,0.773325,0.591541
5,0.35,0.301587,0.301587,1.000000,0.463415,38,88,0,0,0.773325,0.591541
6,0.40,0.301587,0.301587,1.000000,0.463415,38,88,0,0,0.773325,0.591541
7,0.45,0.301587,0.301587,1.000000,0.463415,38,88,0,0,0.773325,0.591541
8,0.50,0.301587,0.301587,1.000000,0.463415,38,88,0,0,0.773325,0.591541
9,0.55,0.301587,0.301587,1.000000,0.463415,38,88,0,0,0.773325,0.591541


,model,student_type,teacher_source,distillation_mode,dataset,threshold,precision,recall,f1,roc_auc,pr_auc,tp,fp,fn,tn,notes
0,rubert-tiny2 student,compact encoder-transformer,teacher_prompt_v2b,soft labels via BCEWithLogitsLoss,valid,0.65,0.404762,0.894737,0.557377,0.773325,0.591541,34,50,4,38,model=cointegrated/rubert-tiny2; threshold selected on valid
1,rubert-tiny2 student,compact encoder-transformer,teacher_prompt_v2b,soft labels via BCEWithLogitsLoss,test,0.65,0.453333,0.894737,0.601770,0.765849,0.603525,34,41,4,47,model=cointegrated/rubert-tiny2; valid-selected threshold applied once to test


## 7. Experiment B: Qwen2.5-0.5B student


Qwen student запускается как sequence classification model. Если GPU недоступен, блок пропускается: для decoder-only модели на CPU эксперимент обычно слишком тяжёлый и может не завершиться за разумное время. Если LoRA через `peft` недоступна или не подключилась, используется fallback с замороженной base model и обучением только classification head.


In [15]:
qwen_valid_metrics = pd.DataFrame()
qwen_test_metrics = pd.DataFrame()
qwen_threshold_table = pd.DataFrame()

qwen_can_run = RUN_QWEN_STUDENT and TRANSFORMERS_AVAILABLE and torch.cuda.is_available()

if not RUN_QWEN_STUDENT:
    skipped_rows.append({"experiment": "Qwen2.5-0.5B student", "status": "skipped", "reason": "RUN_QWEN_STUDENT=False"})
elif not TRANSFORMERS_AVAILABLE:
    skipped_rows.append({"experiment": "Qwen2.5-0.5B student", "status": "skipped", "reason": "torch/transformers are not installed"})
elif not torch.cuda.is_available():
    skipped_rows.append({"experiment": "Qwen2.5-0.5B student", "status": "skipped", "reason": "GPU is not available; Qwen block is intentionally skipped"})
else:
    ensure_distillation_data_loaded()

    qwen_model_name, qwen_tokenizer, qwen_model = load_first_available_sequence_classifier(QWEN_MODEL_CANDIDATES, num_labels=2)
    print("Qwen student model:", qwen_model_name)

    if qwen_tokenizer.pad_token is None:
        qwen_tokenizer.pad_token = qwen_tokenizer.eos_token
    qwen_model.config.pad_token_id = qwen_tokenizer.pad_token_id

    qwen_distillation_mode = "soft labels via BCEWithLogitsLoss"
    qwen_notes = []

    if USE_LORA_FOR_QWEN and PEFT_AVAILABLE:
        try:
            lora_config = LoraConfig(
                task_type=TaskType.SEQ_CLS,
                inference_mode=False,
                r=8,
                lora_alpha=16,
                lora_dropout=0.05,
                target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            )
            qwen_model = get_peft_model(qwen_model, lora_config)
            qwen_notes.append("LoRA enabled")
            if hasattr(qwen_model, "print_trainable_parameters"):
                qwen_model.print_trainable_parameters()
        except Exception as exc:
            qwen_notes.append(f"LoRA failed; classifier-head fallback. Error: {exc}")
            for param in qwen_model.base_model.parameters() if hasattr(qwen_model, "base_model") else qwen_model.parameters():
                param.requires_grad = False
            if hasattr(qwen_model, "score"):
                for param in qwen_model.score.parameters():
                    param.requires_grad = True
            elif hasattr(qwen_model, "classifier"):
                for param in qwen_model.classifier.parameters():
                    param.requires_grad = True
    else:
        qwen_notes.append("LoRA unavailable or disabled; classifier-head fallback")
        for param in qwen_model.parameters():
            param.requires_grad = False
        if hasattr(qwen_model, "score"):
            for param in qwen_model.score.parameters():
                param.requires_grad = True
        elif hasattr(qwen_model, "classifier"):
            for param in qwen_model.classifier.parameters():
                param.requires_grad = True
        else:
            qwen_notes.append("Warning: classification head was not found by common attribute names")

    qwen_train_dataset = NewsRiskDataset(
        qwen_tokenizer,
        teacher_train_df["model_text"],
        labels=teacher_train_df["soft_y"],
        max_length=256,
    )
    qwen_valid_dataset = NewsRiskDataset(
        qwen_tokenizer,
        valid_df["model_text"],
        labels=y_valid,
        max_length=256,
    )
    qwen_test_dataset = NewsRiskDataset(
        qwen_tokenizer,
        test_df["model_text"],
        labels=y_test,
        max_length=256,
    )

    bf16_available = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    fp16_available = torch.cuda.is_available() and not bf16_available

    qwen_args = make_training_args(
        output_dir=MODEL_DIR / "qwen_0_5b_student",
        learning_rate=1e-5,
        train_batch_size=2,
        eval_batch_size=4,
        num_train_epochs=2,
        weight_decay=0.01,
        gradient_accumulation_steps=8,
        fp16=fp16_available,
        bf16=bf16_available,
    )

    qwen_trainer = SoftLabelTrainer(
        model=qwen_model,
        args=qwen_args,
        train_dataset=qwen_train_dataset,
        eval_dataset=qwen_valid_dataset,
        compute_metrics=compute_transformer_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
    )

    qwen_trainer.train()

    qwen_valid_logits = qwen_trainer.predict(qwen_valid_dataset).predictions
    qwen_test_logits = qwen_trainer.predict(qwen_test_dataset).predictions
    qwen_valid_score = positive_scores_from_logits(qwen_valid_logits)
    qwen_test_score = positive_scores_from_logits(qwen_test_logits)

    qwen_threshold, qwen_threshold_table = select_best_threshold_by_f1(y_valid, qwen_valid_score)
    qwen_valid_pred = (qwen_valid_score >= qwen_threshold).astype(int)
    qwen_test_pred = (qwen_test_score >= qwen_threshold).astype(int)

    qwen_valid_metric_dict = compute_binary_metrics(y_valid, qwen_valid_pred, qwen_valid_score)
    qwen_test_metric_dict = compute_binary_metrics(y_test, qwen_test_pred, qwen_test_score)

    append_result_row(
        experiment_rows,
        model="Qwen2.5-0.5B student",
        student_type="small decoder-only LLM sequence classifier",
        distillation_mode=qwen_distillation_mode,
        dataset="valid",
        threshold=qwen_threshold,
        metrics=qwen_valid_metric_dict,
        notes=f"model={qwen_model_name}; " + "; ".join(qwen_notes),
    )
    append_result_row(
        experiment_rows,
        model="Qwen2.5-0.5B student",
        student_type="small decoder-only LLM sequence classifier",
        distillation_mode=qwen_distillation_mode,
        dataset="test",
        threshold=qwen_threshold,
        metrics=qwen_test_metric_dict,
        notes=f"model={qwen_model_name}; valid-selected threshold applied once to test; " + "; ".join(qwen_notes),
    )

    qwen_valid_predictions = valid_df[["sample_id", TARGET_COLUMN]].copy()
    qwen_valid_predictions["student_score"] = qwen_valid_score
    qwen_valid_predictions["student_pred"] = qwen_valid_pred
    qwen_valid_predictions.to_csv(STUDENT_PRED_DIR / "valid_qwen_0_5b_student.csv", index=False)

    qwen_test_predictions = test_df[["sample_id", TARGET_COLUMN]].copy()
    qwen_test_predictions["student_score"] = qwen_test_score
    qwen_test_predictions["student_pred"] = qwen_test_pred
    qwen_test_predictions.to_csv(STUDENT_PRED_DIR / "test_qwen_0_5b_student.csv", index=False)

    qwen_threshold_table.to_csv(ERROR_DIR / "valid_qwen_0_5b_threshold_tuning.csv", index=False)

    qwen_valid_metrics = pd.DataFrame([experiment_rows[-2]])
    qwen_test_metrics = pd.DataFrame([experiment_rows[-1]])
    display(qwen_threshold_table)
    display(pd.concat([qwen_valid_metrics, qwen_test_metrics], ignore_index=True))


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Qwen student model: Qwen/Qwen2.5-0.5B
trainable params: 1,083,136 || all params: 495,117,696 || trainable%: 0.2188


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc,Pr Auc
1,17.424403,0.908299,0.650794,0.363636,0.210526,0.266667,0.558164,0.339102
2,11.466718,0.894868,0.603175,0.363636,0.421053,0.390244,0.567883,0.352947


,threshold,accuracy,precision,recall,f1,tp,fp,fn,tn,roc_auc,pr_auc
0,0.10,0.436508,0.314607,0.736842,0.440945,28,61,10,27,0.567883,0.352947
1,0.15,0.460317,0.321429,0.710526,0.442623,27,57,11,31,0.567883,0.352947
2,0.20,0.492063,0.337500,0.710526,0.457627,27,53,11,35,0.567883,0.352947
3,0.25,0.492063,0.328947,0.657895,0.438596,25,51,13,37,0.567883,0.352947
4,0.30,0.515873,0.323077,0.552632,0.407767,21,44,17,44,0.567883,0.352947
5,0.35,0.571429,0.357143,0.526316,0.425532,20,36,18,52,0.567883,0.352947
6,0.40,0.595238,0.372549,0.500000,0.426966,19,32,19,56,0.567883,0.352947
7,0.45,0.587302,0.360000,0.473684,0.409091,18,32,20,56,0.567883,0.352947
8,0.50,0.603175,0.363636,0.421053,0.390244,16,28,22,60,0.567883,0.352947
9,0.55,0.619048,0.375000,0.394737,0.384615,15,25,23,63,0.567883,0.352947


,model,student_type,teacher_source,distillation_mode,dataset,threshold,precision,recall,f1,roc_auc,pr_auc,tp,fp,fn,tn,notes
0,Qwen2.5-0.5B student,small decoder-only LLM sequence classifier,teacher_prompt_v2b,soft labels via BCEWithLogitsLoss,valid,0.2,0.337500,0.710526,0.457627,0.567883,0.352947,27,53,11,35,model=Qwen/Qwen2.5-0.5B; LoRA enabled
1,Qwen2.5-0.5B student,small decoder-only LLM sequence classifier,teacher_prompt_v2b,soft labels via BCEWithLogitsLoss,test,0.2,0.253012,0.552632,0.347107,0.337919,0.231112,21,62,17,26,model=Qwen/Qwen2.5-0.5B; valid-selected threshold applied once to test; LoRA enabled


In [13]:
# =========================
# Qwen sanity-check:
# hard labels instead of soft teacher labels
# =========================

import gc
import numpy as np
import pandas as pd

# Варианты:
# "manual_alert_flag"  -> обучаем Qwen на настоящем alert_flag train
# "teacher_hard"       -> обучаем Qwen на teacher_label 0/1 без confidence
QWEN_SANITY_TARGET = "manual_alert_flag"   # сначала лучше так

QWEN_SANITY_MODEL_NAME = "Qwen/Qwen2.5-0.5B"
QWEN_SANITY_MAX_LENGTH = 256
QWEN_SANITY_EPOCHS = 3
QWEN_SANITY_LR = 2e-5
QWEN_SANITY_TRAIN_BATCH_SIZE = 2
QWEN_SANITY_EVAL_BATCH_SIZE = 4
QWEN_SANITY_GRAD_ACCUM = 8

qwen_sanity_valid_metrics = pd.DataFrame()
qwen_sanity_test_metrics = pd.DataFrame()
qwen_sanity_threshold_table = pd.DataFrame()

if not TRANSFORMERS_AVAILABLE:
    raise RuntimeError("torch/transformers are not available.")

if not torch.cuda.is_available():
    raise RuntimeError("GPU is not available. Do not run Qwen sanity-check on CPU.")

if not PEFT_AVAILABLE:
    raise RuntimeError("peft is not available. LoRA is required for this sanity-check.")

# Освобождаем память от предыдущего Qwen-запуска
for obj_name in ["qwen_model", "qwen_trainer", "qwen_tokenizer"]:
    if obj_name in globals():
        try:
            del globals()[obj_name]
        except Exception:
            pass

gc.collect()
torch.cuda.empty_cache()

# -------------------------
# Prepare sanity train data
# -------------------------

if QWEN_SANITY_TARGET == "manual_alert_flag":
    qwen_sanity_train_df = train_df[["sample_id", "model_text", TARGET_COLUMN]].copy()
    qwen_sanity_train_df["sanity_y"] = qwen_sanity_train_df[TARGET_COLUMN].astype(float)
    qwen_sanity_source = "manual alert_flag"

elif QWEN_SANITY_TARGET == "teacher_hard":
    qwen_sanity_train_df = teacher_train_df[
        ["sample_id", "model_text", TARGET_COLUMN, "teacher_hard_y"]
    ].copy()
    qwen_sanity_train_df["sanity_y"] = qwen_sanity_train_df["teacher_hard_y"].astype(float)
    qwen_sanity_source = "hard teacher_label"

else:
    raise ValueError("QWEN_SANITY_TARGET must be 'manual_alert_flag' or 'teacher_hard'.")

print("Qwen sanity target:", QWEN_SANITY_TARGET)
print("Qwen sanity source:", qwen_sanity_source)
print("Train rows:", len(qwen_sanity_train_df))
display(qwen_sanity_train_df["sanity_y"].value_counts().rename("sanity_y_count"))

# -------------------------
# Load Qwen model
# -------------------------

qwen_sanity_tokenizer = AutoTokenizer.from_pretrained(QWEN_SANITY_MODEL_NAME)

qwen_sanity_model = AutoModelForSequenceClassification.from_pretrained(
    QWEN_SANITY_MODEL_NAME,
    num_labels=2,
)

if qwen_sanity_tokenizer.pad_token is None:
    qwen_sanity_tokenizer.pad_token = qwen_sanity_tokenizer.eos_token

qwen_sanity_model.config.pad_token_id = qwen_sanity_tokenizer.pad_token_id

if hasattr(qwen_sanity_model.config, "use_cache"):
    qwen_sanity_model.config.use_cache = False

# -------------------------
# LoRA
# -------------------------

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    modules_to_save=["score"],  # важно для Qwen sequence classification head
    bias="none",
)

qwen_sanity_model = get_peft_model(qwen_sanity_model, lora_config)

print("LoRA enabled for Qwen sanity-check")
if hasattr(qwen_sanity_model, "print_trainable_parameters"):
    qwen_sanity_model.print_trainable_parameters()

# -------------------------
# Datasets
# -------------------------

qwen_sanity_train_dataset = NewsRiskDataset(
    qwen_sanity_tokenizer,
    qwen_sanity_train_df["model_text"],
    labels=qwen_sanity_train_df["sanity_y"],
    max_length=QWEN_SANITY_MAX_LENGTH,
)

qwen_sanity_valid_dataset = NewsRiskDataset(
    qwen_sanity_tokenizer,
    valid_df["model_text"],
    labels=y_valid,
    max_length=QWEN_SANITY_MAX_LENGTH,
)

qwen_sanity_test_dataset = NewsRiskDataset(
    qwen_sanity_tokenizer,
    test_df["model_text"],
    labels=y_test,
    max_length=QWEN_SANITY_MAX_LENGTH,
)

# -------------------------
# Training args
# -------------------------

bf16_available = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16_available = torch.cuda.is_available() and not bf16_available

qwen_sanity_args = make_training_args(
    output_dir=MODEL_DIR / f"qwen_0_5b_sanity_{QWEN_SANITY_TARGET}",
    learning_rate=QWEN_SANITY_LR,
    train_batch_size=QWEN_SANITY_TRAIN_BATCH_SIZE,
    eval_batch_size=QWEN_SANITY_EVAL_BATCH_SIZE,
    num_train_epochs=QWEN_SANITY_EPOCHS,
    weight_decay=0.01,
    gradient_accumulation_steps=QWEN_SANITY_GRAD_ACCUM,
    fp16=fp16_available,
    bf16=bf16_available,
)

# SoftLabelTrainer здесь тоже подходит:
# labels у нас hard 0/1, но loss BCEWithLogitsLoss корректно работает с float labels.
qwen_sanity_trainer = SoftLabelTrainer(
    model=qwen_sanity_model,
    args=qwen_sanity_args,
    train_dataset=qwen_sanity_train_dataset,
    eval_dataset=qwen_sanity_valid_dataset,
    compute_metrics=compute_transformer_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

qwen_sanity_trainer.train()

# -------------------------
# Predict valid/test
# -------------------------

qwen_sanity_valid_logits = qwen_sanity_trainer.predict(qwen_sanity_valid_dataset).predictions
qwen_sanity_test_logits = qwen_sanity_trainer.predict(qwen_sanity_test_dataset).predictions

qwen_sanity_valid_score = positive_scores_from_logits(qwen_sanity_valid_logits)
qwen_sanity_test_score = positive_scores_from_logits(qwen_sanity_test_logits)

qwen_sanity_threshold, qwen_sanity_threshold_table = select_best_threshold_by_f1(
    y_valid,
    qwen_sanity_valid_score,
)

qwen_sanity_valid_pred = (qwen_sanity_valid_score >= qwen_sanity_threshold).astype(int)
qwen_sanity_test_pred = (qwen_sanity_test_score >= qwen_sanity_threshold).astype(int)

qwen_sanity_valid_metric_dict = compute_binary_metrics(
    y_valid,
    qwen_sanity_valid_pred,
    qwen_sanity_valid_score,
)

qwen_sanity_test_metric_dict = compute_binary_metrics(
    y_test,
    qwen_sanity_test_pred,
    qwen_sanity_test_score,
)

# -------------------------
# Result rows
# -------------------------

qwen_sanity_notes = (
    f"model={QWEN_SANITY_MODEL_NAME}; "
    f"LoRA enabled; "
    f"sanity target={QWEN_SANITY_TARGET}; "
    f"trained on {qwen_sanity_source}; "
    f"valid-selected threshold applied once to test"
)

append_result_row(
    experiment_rows,
    model=f"Qwen2.5-0.5B sanity-check ({QWEN_SANITY_TARGET})",
    student_type="small decoder-only LLM sequence classifier",
    distillation_mode=f"hard labels sanity-check: {qwen_sanity_source}",
    dataset="valid",
    threshold=qwen_sanity_threshold,
    metrics=qwen_sanity_valid_metric_dict,
    notes=qwen_sanity_notes,
)

append_result_row(
    experiment_rows,
    model=f"Qwen2.5-0.5B sanity-check ({QWEN_SANITY_TARGET})",
    student_type="small decoder-only LLM sequence classifier",
    distillation_mode=f"hard labels sanity-check: {qwen_sanity_source}",
    dataset="test",
    threshold=qwen_sanity_threshold,
    metrics=qwen_sanity_test_metric_dict,
    notes=qwen_sanity_notes,
)

qwen_sanity_valid_metrics = pd.DataFrame([experiment_rows[-2]])
qwen_sanity_test_metrics = pd.DataFrame([experiment_rows[-1]])

# -------------------------
# Save predictions
# -------------------------

qwen_sanity_valid_predictions = valid_df[["sample_id", TARGET_COLUMN]].copy()
qwen_sanity_valid_predictions["student_score"] = qwen_sanity_valid_score
qwen_sanity_valid_predictions["student_pred"] = qwen_sanity_valid_pred
qwen_sanity_valid_predictions.to_csv(
    STUDENT_PRED_DIR / f"valid_qwen_0_5b_sanity_{QWEN_SANITY_TARGET}.csv",
    index=False,
)

qwen_sanity_test_predictions = test_df[["sample_id", TARGET_COLUMN]].copy()
qwen_sanity_test_predictions["student_score"] = qwen_sanity_test_score
qwen_sanity_test_predictions["student_pred"] = qwen_sanity_test_pred
qwen_sanity_test_predictions.to_csv(
    STUDENT_PRED_DIR / f"test_qwen_0_5b_sanity_{QWEN_SANITY_TARGET}.csv",
    index=False,
)

qwen_sanity_threshold_table.to_csv(
    ERROR_DIR / f"valid_qwen_0_5b_sanity_{QWEN_SANITY_TARGET}_threshold_tuning.csv",
    index=False,
)

display(qwen_sanity_threshold_table)
display(pd.concat([qwen_sanity_valid_metrics, qwen_sanity_test_metrics], ignore_index=True))

Qwen sanity target: manual_alert_flag
Qwen sanity source: manual alert_flag
Train rows: 587


,sanity_y_count
sanity_y,
0.0,409
1.0,178


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


LoRA enabled for Qwen sanity-check
trainable params: 1,083,136 || all params: 495,117,696 || trainable%: 0.2188


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc,Pr Auc
1,17.678593,1.665633,0.666667,0.000000,0.000000,0.000000,0.391896,0.250209
2,11.444074,1.171906,0.603175,0.227273,0.131579,0.166667,0.452004,0.275246
3,7.314410,1.125305,0.571429,0.214286,0.157895,0.181818,0.465610,0.282064


,threshold,accuracy,precision,recall,f1,tp,fp,fn,tn,roc_auc,pr_auc
0,0.10,0.428571,0.311111,0.736842,0.437500,28,62,10,26,0.46561,0.282064
1,0.15,0.468254,0.301370,0.578947,0.396396,22,51,16,37,0.46561,0.282064
2,0.20,0.531746,0.333333,0.552632,0.415842,21,42,17,46,0.46561,0.282064
3,0.25,0.547619,0.313725,0.421053,0.359551,16,35,22,53,0.46561,0.282064
4,0.30,0.547619,0.256410,0.263158,0.259740,10,29,28,59,0.46561,0.282064
5,0.35,0.547619,0.256410,0.263158,0.259740,10,29,28,59,0.46561,0.282064
6,0.40,0.555556,0.250000,0.236842,0.243243,9,27,29,61,0.46561,0.282064
7,0.45,0.555556,0.218750,0.184211,0.200000,7,25,31,63,0.46561,0.282064
8,0.50,0.579365,0.222222,0.157895,0.184615,6,21,32,67,0.46561,0.282064
9,0.55,0.587302,0.230769,0.157895,0.187500,6,20,32,68,0.46561,0.282064


,model,student_type,teacher_source,distillation_mode,dataset,threshold,precision,recall,f1,roc_auc,pr_auc,tp,fp,fn,tn,notes
0,Qwen2.5-0.5B sanity-check (manual_alert_flag),small decoder-only LLM sequence classifier,teacher_prompt_v2b,hard labels sanity-check: manual alert_flag,valid,0.1,0.311111,0.736842,0.437500,0.465610,0.282064,28,62,10,26,model=Qwen/Qwen2.5-0.5B; LoRA enabled; sanity target=manual_alert_flag; trained on manual alert_flag; valid-selected threshold applied once to test
1,Qwen2.5-0.5B sanity-check (manual_alert_flag),small decoder-only LLM sequence classifier,teacher_prompt_v2b,hard labels sanity-check: manual alert_flag,test,0.1,0.272727,0.631579,0.380952,0.447967,0.273508,24,64,14,24,model=Qwen/Qwen2.5-0.5B; LoRA enabled; sanity target=manual_alert_flag; trained on manual alert_flag; valid-selected threshold applied once to test


## 8. Final comparison table


In [21]:
# =========================
# 8. Final comparison table without duplicated Qwen runs
# =========================

from collections import OrderedDict
import numpy as np
import pandas as pd

FINAL_COLUMNS = [
    "model",
    "student_type",
    "teacher_source",
    "distillation_mode",
    "dataset",
    "threshold",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc",
    "tp",
    "fp",
    "fn",
    "tn",
    "notes",
    "interpretation",
]

# -------------------------
# Reference rows
# -------------------------

reference_rows = [
    {
        "model": "TF-IDF student v2b final",
        "student_type": "linear TF-IDF student",
        "teacher_source": "GPT-4.1 teacher labels",
        "distillation_mode": "final selected reference",
        "dataset": "valid",
        "threshold": 0.50,
        "precision": 0.738,
        "recall": 0.816,
        "f1": 0.775,
        "roc_auc": 0.910,
        "pr_auc": 0.824,
        "tp": None,
        "fp": None,
        "fn": None,
        "tn": None,
        "notes": "Reference final model from TF-IDF student v2b.",
        "interpretation": "final selected model",
    },
    {
        "model": "TF-IDF student v2b final",
        "student_type": "linear TF-IDF student",
        "teacher_source": "GPT-4.1 teacher labels",
        "distillation_mode": "final selected reference",
        "dataset": "test",
        "threshold": 0.50,
        "precision": 0.773,
        "recall": 0.895,
        "f1": 0.829,
        "roc_auc": 0.969,
        "pr_auc": 0.936,
        "tp": None,
        "fp": None,
        "fn": None,
        "tn": None,
        "notes": "Reference final model from TF-IDF student v2b.",
        "interpretation": "final selected model",
    },
    {
        "model": "rubert-tiny2 previous run",
        "student_type": "compact encoder-transformer student",
        "teacher_source": "GPT-4.1 teacher labels",
        "distillation_mode": "previous transformer student check",
        "dataset": "valid",
        "threshold": None,
        "precision": 0.400,
        "recall": 0.895,
        "f1": 0.553,
        "roc_auc": 0.772,
        "pr_auc": 0.607,
        "tp": None,
        "fp": None,
        "fn": None,
        "tn": None,
        "notes": "Previous rubert-tiny2 run from project context.",
        "interpretation": "additional transformer student check; not selected",
    },
    {
        "model": "rubert-tiny2 previous run",
        "student_type": "compact encoder-transformer student",
        "teacher_source": "GPT-4.1 teacher labels",
        "distillation_mode": "previous transformer student check",
        "dataset": "test",
        "threshold": None,
        "precision": 0.440,
        "recall": 0.868,
        "f1": 0.584,
        "roc_auc": 0.754,
        "pr_auc": 0.585,
        "tp": None,
        "fp": None,
        "fn": None,
        "tn": None,
        "notes": "Previous rubert-tiny2 run from project context.",
        "interpretation": "additional transformer student check; not selected",
    },
]

# Fallback rows only if Qwen was not present in experiment_rows
qwen_colab_rows = [
    {
        "model": "Qwen2.5-0.5B student",
        "student_type": "small decoder-only LLM sequence classifier",
        "teacher_source": "teacher_prompt_v2b",
        "distillation_mode": "soft labels via BCEWithLogitsLoss",
        "dataset": "valid",
        "threshold": 0.15,
        "precision": 0.310924,
        "recall": 0.973684,
        "f1": 0.471338,
        "roc_auc": 0.429575,
        "pr_auc": 0.280056,
        "tp": 37,
        "fp": 82,
        "fn": 1,
        "tn": 6,
        "notes": "Colab GPU run. LoRA enabled: 1,083,136 trainable params of 495,117,696.",
        "interpretation": "small LLM student with LoRA; not selected",
    },
    {
        "model": "Qwen2.5-0.5B student",
        "student_type": "small decoder-only LLM sequence classifier",
        "teacher_source": "teacher_prompt_v2b",
        "distillation_mode": "soft labels via BCEWithLogitsLoss",
        "dataset": "test",
        "threshold": 0.15,
        "precision": 0.308943,
        "recall": 1.000000,
        "f1": 0.472050,
        "roc_auc": 0.420305,
        "pr_auc": 0.272135,
        "tp": 38,
        "fp": 85,
        "fn": 0,
        "tn": 3,
        "notes": "Colab GPU run. Valid-selected threshold applied once to test. LoRA enabled.",
        "interpretation": "small LLM student with LoRA; not selected",
    },
]

qwen_sanity_colab_rows = [
    {
        "model": "Qwen2.5-0.5B sanity-check (manual_alert_flag)",
        "student_type": "small decoder-only LLM sequence classifier",
        "teacher_source": "manual alert_flag",
        "distillation_mode": "hard labels sanity-check: manual alert_flag",
        "dataset": "valid",
        "threshold": 0.10,
        "precision": 0.311111,
        "recall": 0.736842,
        "f1": 0.437500,
        "roc_auc": 0.465610,
        "pr_auc": 0.282064,
        "tp": 28,
        "fp": 62,
        "fn": 10,
        "tn": 26,
        "notes": "Colab GPU sanity-check with Qwen + LoRA trained on manual alert_flag.",
        "interpretation": "diagnostic supervised sanity-check; not selected",
    },
    {
        "model": "Qwen2.5-0.5B sanity-check (manual_alert_flag)",
        "student_type": "small decoder-only LLM sequence classifier",
        "teacher_source": "manual alert_flag",
        "distillation_mode": "hard labels sanity-check: manual alert_flag",
        "dataset": "test",
        "threshold": 0.10,
        "precision": 0.272727,
        "recall": 0.631579,
        "f1": 0.380952,
        "roc_auc": 0.447967,
        "pr_auc": 0.273508,
        "tp": 24,
        "fp": 64,
        "fn": 14,
        "tn": 24,
        "notes": "Colab GPU sanity-check with Qwen + LoRA trained on manual alert_flag.",
        "interpretation": "diagnostic supervised sanity-check; not selected",
    },
]

# -------------------------
# Build raw comparison table
# -------------------------

comparison_frames = [pd.DataFrame(reference_rows)]

experiment_results = (
    pd.DataFrame(experiment_rows)
    if "experiment_rows" in globals()
    else pd.DataFrame()
)

if not experiment_results.empty:
    experiment_results = experiment_results.copy()

    for column in FINAL_COLUMNS:
        if column not in experiment_results.columns:
            experiment_results[column] = None

    experiment_results.loc[
        experiment_results["model"].astype(str).str.contains("rubert-tiny2", case=False, na=False),
        "interpretation",
    ] = "additional transformer student check; not selected"

    experiment_results.loc[
        experiment_results["model"].astype(str).str.contains("Qwen2.5-0.5B student", case=False, na=False),
        "interpretation",
    ] = "small LLM student with LoRA; not selected"

    experiment_results.loc[
        experiment_results["model"].astype(str).str.contains("sanity-check", case=False, na=False),
        "interpretation",
    ] = "diagnostic supervised sanity-check; not selected"

    comparison_frames.append(experiment_results[FINAL_COLUMNS])

small_llm_student_comparison = pd.concat(comparison_frames, ignore_index=True)

# Add fallback Qwen rows only if no Qwen rows exist at all
existing_models = set(small_llm_student_comparison["model"].dropna().astype(str))

if "Qwen2.5-0.5B student" not in existing_models:
    small_llm_student_comparison = pd.concat(
        [small_llm_student_comparison, pd.DataFrame(qwen_colab_rows)],
        ignore_index=True,
    )

existing_models = set(small_llm_student_comparison["model"].dropna().astype(str))

if "Qwen2.5-0.5B sanity-check (manual_alert_flag)" not in existing_models:
    small_llm_student_comparison = pd.concat(
        [small_llm_student_comparison, pd.DataFrame(qwen_sanity_colab_rows)],
        ignore_index=True,
    )

for column in FINAL_COLUMNS:
    if column not in small_llm_student_comparison.columns:
        small_llm_student_comparison[column] = None

small_llm_student_comparison = small_llm_student_comparison[FINAL_COLUMNS].copy()

# -------------------------
# Clean duplicated rows
# -------------------------

def keep_best_qwen_soft_label_run(df: pd.DataFrame) -> pd.DataFrame:
    """
    Keep only one valid/test pair for Qwen2.5-0.5B student.
    If several Qwen soft-label runs exist, choose the run with the best valid F1
    and keep the test row with the same threshold.
    """
    df = df.copy()

    qwen_model = "Qwen2.5-0.5B student"
    qwen_mask = df["model"].eq(qwen_model)

    qwen_df = df[qwen_mask].copy()
    other_df = df[~qwen_mask].copy()

    if qwen_df.empty:
        return df

    qwen_df["f1_num"] = pd.to_numeric(qwen_df["f1"], errors="coerce")
    qwen_df["threshold_num"] = pd.to_numeric(qwen_df["threshold"], errors="coerce")

    valid_qwen = qwen_df[qwen_df["dataset"].eq("valid")].copy()

    if valid_qwen.empty:
        qwen_df = qwen_df.drop(columns=["f1_num", "threshold_num"], errors="ignore")
        return pd.concat([other_df, qwen_df], ignore_index=True)

    best_valid_idx = valid_qwen["f1_num"].idxmax()
    best_threshold = qwen_df.loc[best_valid_idx, "threshold_num"]

    if pd.isna(best_threshold):
        best_qwen_df = qwen_df.loc[[best_valid_idx]].copy()
    else:
        threshold_mask = np.isclose(
            qwen_df["threshold_num"].astype(float),
            float(best_threshold),
            equal_nan=False,
        )
        best_qwen_df = qwen_df[
            threshold_mask
            & qwen_df["dataset"].isin(["valid", "test"])
        ].copy()

    # Keep one row per dataset if duplicates remain
    dataset_rank = {"valid": 0, "test": 1}
    best_qwen_df["_dataset_rank"] = best_qwen_df["dataset"].map(dataset_rank).fillna(99)

    best_qwen_df = (
        best_qwen_df
        .sort_values(["_dataset_rank", "f1_num"], ascending=[True, False])
        .drop_duplicates(subset=["model", "dataset"], keep="first")
        .drop(columns=["f1_num", "threshold_num", "_dataset_rank"], errors="ignore")
    )

    return pd.concat([other_df, best_qwen_df], ignore_index=True)


def drop_exact_metric_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove exact duplicates by model/dataset/threshold/metrics.
    Keeps the first occurrence.
    """
    duplicate_subset = [
        "model",
        "dataset",
        "threshold",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "pr_auc",
    ]

    return df.drop_duplicates(subset=duplicate_subset, keep="first").reset_index(drop=True)


small_llm_student_comparison = drop_exact_metric_duplicates(small_llm_student_comparison)
small_llm_student_comparison = keep_best_qwen_soft_label_run(small_llm_student_comparison)

# -------------------------
# Final ordering
# -------------------------

model_order = OrderedDict([
    ("TF-IDF student v2b final", 0),
    ("rubert-tiny2 previous run", 1),
    ("rubert-tiny2 student", 2),
    ("Qwen2.5-0.5B student", 3),
    ("Qwen2.5-0.5B sanity-check (manual_alert_flag)", 4),
])

dataset_order = {"valid": 0, "test": 1}

small_llm_student_comparison["_model_order"] = (
    small_llm_student_comparison["model"].map(model_order).fillna(99)
)
small_llm_student_comparison["_dataset_order"] = (
    small_llm_student_comparison["dataset"].map(dataset_order).fillna(99)
)

small_llm_student_comparison = (
    small_llm_student_comparison
    .sort_values(["_model_order", "_dataset_order", "model", "dataset"])
    .drop(columns=["_model_order", "_dataset_order"])
    .reset_index(drop=True)
)

# -------------------------
# Save and display
# -------------------------

small_llm_student_comparison.to_csv(
    ERROR_DIR / "small_llm_student_comparison.csv",
    index=False,
)

display(small_llm_student_comparison)

# -------------------------
# Skipped experiments
# -------------------------

skipped_experiments = (
    pd.DataFrame(skipped_rows)
    if "skipped_rows" in globals()
    else pd.DataFrame()
)

if not skipped_experiments.empty:
    real_qwen_metrics_exist = small_llm_student_comparison["model"].eq(
        "Qwen2.5-0.5B student"
    ).any()

    if real_qwen_metrics_exist and "experiment" in skipped_experiments.columns:
        skipped_experiments = skipped_experiments[
            ~skipped_experiments["experiment"]
            .astype(str)
            .str.contains("Qwen2.5-0.5B student", case=False, na=False)
        ].reset_index(drop=True)

if skipped_experiments.empty:
    print("No skipped experiments after Colab metrics reconciliation.")
else:
    display(skipped_experiments)
    skipped_experiments.to_csv(
        ERROR_DIR / "small_llm_student_skipped_experiments.csv",
        index=False,
    )

,model,student_type,teacher_source,distillation_mode,dataset,threshold,precision,recall,f1,roc_auc,pr_auc,tp,fp,fn,tn,notes,interpretation
0,TF-IDF student v2b final,linear TF-IDF student,GPT-4.1 teacher labels,final selected reference,valid,0.50,0.738000,0.816000,0.775000,0.910000,0.824000,None,None,None,None,Reference final model from TF-IDF student v2b.,final selected model
1,TF-IDF student v2b final,linear TF-IDF student,GPT-4.1 teacher labels,final selected reference,test,0.50,0.773000,0.895000,0.829000,0.969000,0.936000,None,None,None,None,Reference final model from TF-IDF student v2b.,final selected model
2,rubert-tiny2 previous run,compact encoder-transformer student,GPT-4.1 teacher labels,previous transformer student check,valid,NaN,0.400000,0.895000,0.553000,0.772000,0.607000,None,None,None,None,Previous rubert-tiny2 run from project context.,additional transformer student check; not selected
3,rubert-tiny2 previous run,compact encoder-transformer student,GPT-4.1 teacher labels,previous transformer student check,test,NaN,0.440000,0.868000,0.584000,0.754000,0.585000,None,None,None,None,Previous rubert-tiny2 run from project context.,additional transformer student check; not selected
4,rubert-tiny2 student,compact encoder-transformer,teacher_prompt_v2b,soft labels via BCEWithLogitsLoss,valid,0.65,0.404762,0.894737,0.557377,0.773325,0.591541,34,50,4,38,model=cointegrated/rubert-tiny2; threshold selected on valid,additional transformer student check; not selected
5,rubert-tiny2 student,compact encoder-transformer,teacher_prompt_v2b,soft labels via BCEWithLogitsLoss,test,0.65,0.453333,0.894737,0.601770,0.765849,0.603525,34,41,4,47,model=cointegrated/rubert-tiny2; valid-selected threshold applied once to test,additional transformer student check; not selected
6,Qwen2.5-0.5B student,small decoder-only LLM sequence classifier,teacher_prompt_v2b,soft labels via BCEWithLogitsLoss,valid,0.20,0.337500,0.710526,0.457627,0.567883,0.352947,27,53,11,35,model=Qwen/Qwen2.5-0.5B; LoRA enabled,small LLM student with LoRA; not selected
7,Qwen2.5-0.5B student,small decoder-only LLM sequence classifier,teacher_prompt_v2b,soft labels via BCEWithLogitsLoss,test,0.20,0.253012,0.552632,0.347107,0.337919,0.231112,21,62,17,26,model=Qwen/Qwen2.5-0.5B; valid-selected threshold applied once to test; LoRA enabled,small LLM student with LoRA; not selected
8,Qwen2.5-0.5B sanity-check (manual_alert_flag),small decoder-only LLM sequence classifier,teacher_prompt_v2b,hard labels sanity-check: manual alert_flag,valid,0.10,0.311111,0.736842,0.437500,0.465610,0.282064,28,62,10,26,model=Qwen/Qwen2.5-0.5B; LoRA enabled; sanity target=manual_alert_flag; trained on manual alert_flag; valid-selected threshold applied once to test,diagnostic supervised sanity-check; not selected
9,Qwen2.5-0.5B sanity-check (manual_alert_flag),small decoder-only LLM sequence classifier,teacher_prompt_v2b,hard labels sanity-check: manual alert_flag,test,0.10,0.272727,0.631579,0.380952,0.447967,0.273508,24,64,14,24,model=Qwen/Qwen2.5-0.5B; LoRA enabled; sanity target=manual_alert_flag; trained on manual alert_flag; valid-selected threshold applied once to test,diagnostic supervised sanity-check; not selected


No skipped experiments after Colab metrics reconciliation.


## 9. Выводы

В этом notebook были дополнительно проверены более выразительные student-модели для задачи teacher–student distillation в классификации банковских риск-сигналов. В качестве основной reference-схемы используется `TF-IDF student v2b final`, так как именно она показала наиболее устойчивое качество: на valid F1 около `0.775`, на test F1 около `0.829`, при высоких ROC-AUC и PR-AUC.

`rubert-tiny2` рассматривался как compact encoder-transformer student. Он обучался на тех же входных текстах `title + entity_norm + text_fragment` и использовал teacher labels / teacher confidence только на train-части. В текущем запуске `rubert-tiny2` показал F1 около `0.557` на valid и около `0.602` на test. Это подтверждает, что compact transformer student способен находить часть риск-сигналов, но по качеству всё ещё заметно уступает финальной TF-IDF student-модели.

`Qwen2.5-0.5B` был проверен как small decoder-only LLM student с LoRA. Этот эксперимент ближе к полноценной LLM-distillation, так как student сам является небольшой языковой моделью. Qwen был успешно запущен с LoRA, то есть обучалась только небольшая доля параметров модели, а не выполнялся полный fine-tuning. Для финального сравнения оставлен лучший soft-label Qwen-запуск по valid F1. Даже в этом варианте качество оказалось слабым: valid F1 около `0.457`, test F1 около `0.347`, ROC-AUC и PR-AUC существенно ниже, чем у TF-IDF student v2b. Это означает, что small LLM student в текущем setup не дал улучшения качества.

Дополнительно был проведён sanity-check для Qwen, где модель обучалась не на soft teacher labels, а на ручной train-разметке `alert_flag`. Этот эксперимент нужен был, чтобы проверить, связана ли деградация только с teacher-разметкой. Sanity-check также не дал устойчивого качества: valid F1 около `0.438`, test F1 около `0.381`, ROC-AUC остался ниже `0.5`. Следовательно, проблема связана не только с soft labels, но и с ограниченным размером train-выборки, нестабильностью decoder-only LLM в classification setup и необходимостью более тонкой настройки LoRA/SFT.

Все эксперименты методически разделены: teacher labels и teacher confidence используются только на train, valid применяется для выбора threshold по F1, а test используется только для финальной оценки с уже выбранным threshold. Valid/test не используются для teacher-разметки или обучения student-моделей.

Итоговый вывод: более сложные student-архитектуры сами по себе не гарантируют улучшения качества при малом размере обучающей выборки. В текущем проекте наиболее стабильной и качественной остаётся `TF-IDF student v2b final`. `rubert-tiny2` и `Qwen2.5-0.5B + LoRA` рассматриваются как дополнительные исследовательские проверки, но не выбираются как финальные модели. Для развития neural / small LLM student-подхода потребуется больший объём обучающих данных, калибровка teacher confidence, более аккуратная настройка loss-функции, class balancing, learning rate schedule и LoRA-конфигурации.

